# Serviço Florestal Brasileiro (SFB)

O Serviço Florestal Brasileiro (SFB) é uma autarquia federal vinculada atualmente ao Ministério do Meio Ambiente e Mudança do Clima (MMA). Sua principal função é promover o uso sustentável das florestas públicas brasileiras, buscando conciliar conservação ambiental, geração de renda e desenvolvimento social, especialmente em áreas de forte presença de recursos florestais. Principais atribuições do SFB:

- **Gestão de florestas públicas federais**: Planeja e executa o manejo florestal sustentável, por meio da concessão de florestas públicas a empresas ou cooperativas, com o objetivo de exploração racional dos recursos madeireiros e não madeireiros.
- **Cadastro Ambiental Rural (CAR)**: É o órgão responsável por coordenar o Sistema Nacional de Cadastro Ambiental Rural (SICAR), ferramenta essencial para o controle, monitoramento e combate ao desmatamento, além de apoio à regularização ambiental de imóveis rurais.
- **Fomento ao conhecimento e manejo florestal**: Produz estudos, pesquisas e informações técnicas sobre florestas brasileiras, uso da terra e atividades de manejo sustentável.
- **Apoio à cadeia produtiva florestal**: Estimula boas práticas de manejo, apoio a comunidades locais e uso sustentável da biodiversidade florestal.

<br>

O SFB foi criado em 2006 pela Lei de Gestão de Florestas Públicas (Lei nº 11.284/2006) e tem um papel central na Política Nacional de Florestas, atuando como elo entre conservação ambiental e desenvolvimento econômico.

<br>

---

Especificamente sobre o Cadastro Ambiental Rural (CAR), o SFB mantem o _site_ para a consulta pública dos imóveis rurais do Brasil.

> https://consultapublica.car.gov.br/publico/imoveis/index


In [ ]:
import nntplib
import ssl
from urllib.request import urlopen

import geopandas as gpd
import requests
import urllib3
from owslib.util import Authentication
from owslib.wfs import WebFeatureService
from urllib3.util.ssl_ import create_urllib3_context

<br>

-----

## Requests

In [ ]:
url = 'https://geoserver.car.gov.br/geoserver/ows'
# url = 'https://geoserver.car.gov.br/geoserver/ows?service=WFS&acceptversions=2.0.0&request=GetCapabilities'

<br>

Inicialmente tentamos acessar a url, mesmo sem a verificação do SSL e tomo erro.

In [ ]:
r = requests.get(
    url='https://geoserver.car.gov.br/geoserver/ows',
    # context=context,
    # Ignora SSL (não recomendado)
    verify=False,
)
print(r.status_code)

<br>

Sessão

In [ ]:
s = requests.Session()
response = s.get('https://geoserver.car.gov.br/geoserver/ows')

ctx = ssl.create_default_context()

In [ ]:
ctx = ssl.create_default_context()

<br>

---

## Cipher

Por meio do comando abaixo eu descubro que o cipher usado na conexão com o https://geoserver.car.gov.br/geoserver/web utiliza o cipher `AES256-GCM-SHA384`.

```shell
openssl s_client -connect geoserver.car.gov.br:443
openssl s_client -connect geoserver.car.gov.br:443 -state -debug


openssl s_client -showcerts -servername "geoserver.car.gov.br" -connect "geoserver.car.gov.br:443" > cacert.pem

echo quit | openssl s_client -showcerts -servername "curl.haxx.se" -connect curl.haxx.se:443 > cacert.pem


openssl s_client -connect geoserver.car.gov.br:443 -servername geoserver.car.gov.br
```

<br>

Veja o resultado abaixo:

- New, TLSv1.2, Cipher is AES256-GCM-SHA384
- Protocol: TLSv1.2


In [ ]:
!openssl ciphers

In [ ]:
context = ssl.create_default_context()
context.set_ciphers('AES256-GCM-SHA384')

In [ ]:
url = 'geoserver.car.gov.br'
nntp = nntplib.NNTP_SSL(host=url, ssl_context=context, timeout=10)

In [ ]:
aa = urlopen(
    url=url,
    # context=context,
    # cafile=str(pem_file),
    timeout=5,
)

In [ ]:
aa = create_urllib3_context()
aa.get_ciphers()

In [ ]:
for item in create_urllib3_context().get_ciphers():
    print(item['name'])

In [ ]:
http = urllib3.PoolManager()
response = http.request(
    method='GET',
    url=url,
    # ca_certs="CERT_NONE"
)

https://github.com/geopython/OWSLib/issues/1008

In [ ]:
url = 'https://geoserver.car.gov.br/geoserver/ows'

r = requests.get(
    url,
    verify=False,
)
print(r.status_code)

In [ ]:
pem_files = [
    Path('.').absolute() / 'car-gov-br.pem',
    Path('.').absolute() / 'car-gov-br2.pem',
    # Path('.').absolute() / 'car-gov-br-chain.pem',
    Path('.').absolute() / 'car-gov-br-chain3.pem',
]


# context = ssl._create_unverified_context()
for pem_file in pem_files:
    try:
        r = requests.get(url, verify=pem_file.as_posix())
        print(f'Response status code for {pem_file}: {r.status_code}')
    except Exception as e:
        print(f'Erro ao acessar {pem_file}: {e}')
        continue


for pem_file in pem_files:
    # Cria contexto
    context = ssl.create_default_context(cafile=pem_file)

    context = ssl.create_default_context()
    context.options |= ssl.OP_NO_SSLv2
    context.options |= ssl.OP_NO_SSLv3    

    try:
        aa = urlopen(
            url=url,
            # 'https://geoserver.car.gov.br/geoserver/ows?service=WFS&version=1.0.0&request=GetCapabilities',
            context=context,
            # cafile=str(pem_file),
            timeout=5,
        )

        print(aa)
    except Exception as e:
        print(f'Erro ao acessar {pem_file}: {e}')
        continue

In [ ]:
# aa = urlopen(
#     'https://geoserver.car.gov.br/geoserver/ows?service=WFS&version=1.0.0&request=GetCapabilities',
#     context=ssl._create_unverified_context(),
#     # cafile=str(pem_file),
#     timeout=5,
# )

<br>

-----

## OWSLib

In [ ]:
auth = Authentication(
    # cert=pem_file.as_posix(),
    verify=False,
)

auth.urlopen_kwargs

In [ ]:
url = 'https://geoserver.car.gov.br/geoserver/sicar/wfs'

In [ ]:
wfs = WebFeatureService(
    url=url,
    version='2.0.0',
    # verify=True,
    # session=session,
    # auth=Authentication(
    #     # cert=pem_file.as_posix(),
    #     verify=False
    # ),
)

In [ ]:
wfs = WebFeatureService(
    url=url,
    version='2.0.0',
    # verify=True,
    # session=session,
    # auth=Authentication(
    #     # cert=pem_file.as_posix(),
    #     verify=False
    # ),
)

In [ ]:
ssl.get_default_verify_paths()

<br>


----

## Listas *Layers*

In [ ]:
list(wfs.contents)

<br>

-----

## Obter *Layers*

In [ ]:
# Obter os dados no formato GeoJSON (ou outro formato suportado)
response = wfs.getfeature(
    typename='SGDGeo:georegiao',
    # bbox=(173700, 440400, 178700, 441400),
    # srsname='EPSG:28992'
    # srsname='EPSG:4326',
    srsname='EPSG:4674',
    outputFormat='application/json',
)
response

<br>

----

## Geopandas

In [ ]:
gdf = gpd.read_file(filename=response)
gdf.crs

In [ ]:
gdf.info()

In [ ]:
gdf.explore()